human lung cancer

导入相关包

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [2]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/human_lung_cancer/SMI_Lung.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_Nicheformer'
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/nicheformer_human_lung_cancer_emb.parquet"


读取嵌入

In [3]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [4]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial', 'X_Nicheformer'

In [5]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    uns: 'neighbors'
    obsm: 'spatial', 'X_Nicheformer'
    obsp: 'distances', 'connectivities'

In [6]:
max_value = 0
metrics = {"method": "Nicheformer", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
save_label_df = None

In [7]:

for used_res in res_list:
    sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
    true_labels = np.array(adata.obs[celltype_col])
    cluster_labels = np.array(adata.obs[key_added])
    FMI = fowlkes_mallows_score(true_labels, cluster_labels)
    homogeneity = homogeneity_score(true_labels, cluster_labels)
    v_measure = v_measure_score(true_labels, cluster_labels)
    ami = adjusted_mutual_info_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(true_labels, cluster_labels)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

    n_cluster = len(adata.obs[key_added].unique())
    n_celltype = len(adata.obs[celltype_col].unique())
    if mean_value > max_value:
        save_label_df = adata.obs[key_added]
        metrics["ARI"] = ari
        metrics["NMI"] = nmi
        metrics["AMI"] = ami
        metrics["HS"] = homogeneity
        metrics["VM"] = v_measure
        metrics["FMI"] = FMI
        metrics["mean value"] = mean_value
        max_value = mean_value

    print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

/tmp/ipykernel_2628241/737231793.py:2: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)


resolution = 0.1 | n_celltype = 18 | n_cluster = 5 | mean_value = 0.253 | ARI = 0.086 |  NMI = 0.275 | AMI = 0.275 | HS = 0.195 | VM = 0.275 | FMI = 0.412 
resolution = 0.2 | n_celltype = 18 | n_cluster = 6 | mean_value = 0.271 | ARI = 0.071 |  NMI = 0.314 | AMI = 0.313 | HS = 0.242 | VM = 0.314 | FMI = 0.371 
resolution = 0.3 | n_celltype = 18 | n_cluster = 9 | mean_value = 0.313 | ARI = 0.154 |  NMI = 0.351 | AMI = 0.351 | HS = 0.341 | VM = 0.351 | FMI = 0.33 
resolution = 0.4 | n_celltype = 18 | n_cluster = 10 | mean_value = 0.299 | ARI = 0.173 |  NMI = 0.322 | AMI = 0.321 | HS = 0.326 | VM = 0.322 | FMI = 0.332 
resolution = 0.5 | n_celltype = 18 | n_cluster = 10 | mean_value = 0.325 | ARI = 0.206 |  NMI = 0.345 | AMI = 0.345 | HS = 0.351 | VM = 0.345 | FMI = 0.359 
resolution = 0.6 | n_celltype = 18 | n_cluster = 13 | mean_value = 0.318 | ARI = 0.203 |  NMI = 0.333 | AMI = 0.333 | HS = 0.356 | VM = 0.333 | FMI = 0.35 
resolution = 0.7 | n_celltype = 18 | n_cluster = 16 | mean_valu

In [8]:
save_label_df

unique_ID
1-2        0
1-3        0
1-4        6
1-6        6
1-7        0
          ..
20-4048    3
20-4049    3
20-4050    3
20-4051    3
20-4052    0
Name: leiden, Length: 87589, dtype: category
Categories (10, object): ['0', '1', '2', '3', ..., '6', '7', '8', '9']

In [9]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/lung_cancer/nicheformer_human_lung_cancer_labels_repeat_{repeat_times}.csv"

In [10]:
save_label_df.to_csv(save_label_df_path)

In [11]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [12]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_repeat_{repeat_times}.csv"

In [13]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))